# Evaluación final — Reseñas de vinos

**Módulo 06 — Fundamentos de redes neuronales.** Resolución del cuestionario final sobre el dataset `winemag-data-130k-v2.csv`. El enunciado completo está en [`enunciado.md`](enunciado.md).

## Cómo usar esta notebook

El dataset **no está versionado** en el repositorio (los archivos de datos pesados están en el `.gitignore`). Antes de ejecutar, descargalo del repositorio de GitHub del curso y dejalo en:

```
practica/evaluacion-final/datasets/winemag-data-130k-v2.csv
```

## Plan de trabajo

1. Carga y exploración inicial.
2. Copia del original y eliminación de columnas.
3. `data_eliminados`: descarte de nulos y codificación de etiquetas.
4. `data_imputados`: imputación de `points` y `price`, descarte del resto de nulos y codificación.
5. División 70/30 con semilla 17 en ambos conjuntos.
6. Modelos y evaluación, a medida que el cuestionario los pida.

## 1. Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

RANDOM_STATE = 17   # semilla pedida por el enunciado
TEST_SIZE = 0.3     # division 70/30

pd.set_option('display.max_columns', None)

## 2. Carga del dataset

Se carga el archivo original y se lo deja intacto en `datos`. Todo el trabajo posterior se hace sobre copias.

In [ ]:
RUTA = 'datasets/winemag-data-130k-v2.csv'

datos = pd.read_csv(RUTA)
print(f'Filas: {datos.shape[0]:,} | Columnas: {datos.shape[1]}')
datos.head()

### Exploración inicial

Antes de tocar nada conviene ver los tipos de dato y el panorama de valores ausentes: de eso dependen las dos estrategias que compara el cuestionario.

In [ ]:
datos.info()

In [ ]:
nulos = datos.isnull().sum().sort_values(ascending=False)
pd.DataFrame({
    'nulos': nulos,
    'porcentaje': (nulos / len(datos) * 100).round(2)
})

## 3. Copia y eliminación de columnas

Según el enunciado: copia profunda del original y descarte de `region_2`, `taster_twitter_handle`, `designation` y `Unnamed: 0`.

`copy(deep=True)` crea una copia independiente: modificar `datos_copy` no altera `datos`.

In [ ]:
datos_copy = datos.copy(deep = True)

columnas_a_eliminar = ['region_2', 'taster_twitter_handle', 'designation', 'Unnamed: 0']
datos_copy = datos_copy.drop(columns=columnas_a_eliminar)

print('Columnas restantes:', list(datos_copy.columns))
print(f'Dimensiones: {datos_copy.shape}')

## 4. `data_eliminados`

Estrategia de **descarte**: se eliminan todas las filas que tengan algún valor ausente y después se codifican las variables categóricas.

El orden importa: primero se descartan los nulos, después se codifica. Codificar antes obligaría a decidir qué hacer con los `NaN` dentro del encoder.

In [ ]:
data_eliminados = datos_copy.dropna(axis=0)

print(f'Antes : {datos_copy.shape[0]:,} filas')
print(f'Despues: {data_eliminados.shape[0]:,} filas')
print(f'Se descarto el {(1 - len(data_eliminados)/len(datos_copy))*100:.2f}% de las filas')

### Codificación de etiquetas

La **codificación de etiquetas** (`LabelEncoder`) asigna un entero a cada categoría. Se aplica a todas las columnas de tipo objeto.

> Nota: esta codificación introduce un orden numérico que no existe entre las categorías (que Argentina sea 1 y Australia 2 no significa que Australia sea "mayor"). Los modelos basados en árboles lo toleran bien; los modelos lineales y las redes neuronales pueden interpretarlo como una magnitud. El enunciado pide explícitamente esta codificación, así que se usa esta.

In [ ]:
data_eliminados = data_eliminados.copy()

# Se codifican solo las columnas categoricas que se van a usar como feature o target.
# 'description' y 'title' son texto libre de altisima cardinalidad (casi un valor
# distinto por fila): codificarlas con etiquetas no aporta nada y solo gasta tiempo.
columnas_categoricas = ['country', 'province', 'region_1', 'taster_name', 'variety', 'winery']

encoders_eliminados = {}
for col in columnas_categoricas:
    le = LabelEncoder()
    data_eliminados[col] = le.fit_transform(data_eliminados[col])
    encoders_eliminados[col] = le

print('Codificadas:', columnas_categoricas)
data_eliminados.head()

### Separación de features y target

Target: `country`. Es un problema de **clasificación multiclase**.

In [ ]:
features = ['price', 'points', 'province', 'region_1','taster_name','variety', 'winery']
target = 'country'

X_eliminados = data_eliminados[features]
y_eliminados = data_eliminados[target]

print(f'X_eliminados: {X_eliminados.shape}')
print(f'y_eliminados: {y_eliminados.shape} | clases distintas: {y_eliminados.nunique()}')

## 5. `data_imputados`

Estrategia de **imputación**: se completan `points` con su mediana y `price` con su media, y recién después se descartan las filas que sigan teniendo ausentes en otras columnas.

- **Mediana** para `points`: es robusta a valores extremos.
- **Media** para `price`: es la que pide el enunciado.

> **Observación metodológica.** El target de este conjunto es `price`, y el enunciado pide imputarlo con la media antes de separarlo. Imputar la variable objetivo introduce filas cuyo valor real es desconocido y que el modelo aprende como si fueran observaciones válidas — se le enseña a predecir la media. Es una decisión discutible, pero es lo que pide la consigna, así que se sigue al pie de la letra. Vale tenerlo presente al interpretar las métricas.

In [ ]:
data_imputados = datos_copy.copy(deep = True)

mediana_points = data_imputados['points'].median()
media_price = data_imputados['price'].mean()

print(f'Mediana de points: {mediana_points}')
print(f'Media de price   : {media_price:.4f}')

data_imputados['points'] = data_imputados['points'].fillna(mediana_points)
data_imputados['price'] = data_imputados['price'].fillna(media_price)

print(f"\nNulos restantes en points: {data_imputados['points'].isnull().sum()}")
print(f"Nulos restantes en price : {data_imputados['price'].isnull().sum()}")

Ahora sí, se descartan las filas que sigan teniendo ausentes en el resto de las columnas.

In [ ]:
antes = data_imputados.shape[0]
data_imputados = data_imputados.dropna(axis=0)

print(f'Antes : {antes:,} filas')
print(f'Despues: {data_imputados.shape[0]:,} filas')
print(f'\nComparacion con la estrategia de descarte:')
print(f'  data_eliminados: {data_eliminados.shape[0]:,} filas')
print(f'  data_imputados : {data_imputados.shape[0]:,} filas')

### Codificación de etiquetas

In [ ]:
data_imputados = data_imputados.copy()

columnas_categoricas_imp = ['country', 'province', 'region_1', 'taster_name', 'variety', 'winery']

encoders_imputados = {}
for col in columnas_categoricas_imp:
    le = LabelEncoder()
    data_imputados[col] = le.fit_transform(data_imputados[col])
    encoders_imputados[col] = le

print('Codificadas:', columnas_categoricas_imp)
data_imputados.head()

### Separación de features y target

Target: `price`. Es un problema de **regresión**.

In [ ]:
features = ['country', 'points', 'province', 'region_1','taster_name','variety', 'winery']
target = 'price'

X_imputados = data_imputados[features]
y_imputados = data_imputados[target]

print(f'X_imputados: {X_imputados.shape}')
print(f'y_imputados: {y_imputados.shape}')
y_imputados.describe()

## 6. División de los conjuntos

70/30 con `random_state=17` en ambos casos, tal como pide el enunciado.

In [ ]:
# Conjunto de descarte -> clasificacion de country
X_train_el, X_test_el, y_train_el, y_test_el = train_test_split(
    X_eliminados, y_eliminados, test_size=TEST_SIZE, random_state=RANDOM_STATE)

# Conjunto imputado -> regresion de price
X_train_im, X_test_im, y_train_im, y_test_im = train_test_split(
    X_imputados, y_imputados, test_size=TEST_SIZE, random_state=RANDOM_STATE)

print('ELIMINADOS (clasificacion de country)')
print(f'  train: {X_train_el.shape[0]:,} | test: {X_test_el.shape[0]:,}')
print('IMPUTADOS (regresion de price)')
print(f'  train: {X_train_im.shape[0]:,} | test: {X_test_im.shape[0]:,}')

## 7. Modelos y evaluación

A completar con las consignas del cuestionario.

Todo lo anterior deja listo el punto de partida:

| Variable | Contenido |
|---|---|
| `X_train_el`, `X_test_el`, `y_train_el`, `y_test_el` | conjunto de **descarte** — clasificación de `country` |
| `X_train_im`, `X_test_im`, `y_train_im`, `y_test_im` | conjunto **imputado** — regresión de `price` |
| `encoders_eliminados`, `encoders_imputados` | los `LabelEncoder` de cada columna, para revertir la codificación con `inverse_transform` |

Recordatorios para cuando se entrenen los modelos:

- Mantener `random_state=17` en cada estimador, o los resultados no van a ser reproducibles.
- Si se usan redes neuronales o cualquier modelo basado en distancias, **escalar** los datos: ajustar el `StandardScaler` solo con train, o mejor, meterlo en un `Pipeline`.
- En clasificación de `country` las clases están muy desbalanceadas: la exactitud sola puede engañar, conviene mirar también la matriz de confusión y el F1.